## PyTorch解决垃圾二分类问题

PyTorch的HelloWorld级demo， 旨在把tensorflow的知识类比迁移到Torch

### 1. 数据加载与增强

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models
from torchvision import transforms

# 1. 定义总的数据增强/预处理
# 注意：验证集通常不需要 RandomFlip 和 Affine，只需要 Resize 和 Normalize
base_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 2. 加载原始全量数据
full_dataset = datasets.ImageFolder(root='./data/train', transform=base_transform)

# 3. 计算划分数量 (比如 80% 训练, 20% 验证)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

# 4. 执行划分
train_data, val_data = random_split(full_dataset, [train_size, val_size])

# 5. 创建对应的 DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False) # 验证集不需要打乱

### 2. 模型构建与微调设置 (Model & Fine-tuning)

In [ ]:
import torch.nn as nn

# 加载预训练的 VGG16
model = models.vgg16(weights='IMAGENET1K_V1')

# 1. 冻结所有层 (对应 layer.trainable = False)
for param in model.parameters():
    param.requires_grad = False

# 2. 解冻特定的层 (对应你的 block5_conv3 部分)
# VGG16 的 features 里的第 28 层通常对应 block5_conv3
for param in model.features[28:].parameters():
    param.requires_grad = True

# 3. 修改分类头 (对应你的 Sequential 后面加 Dense)
num_features = model.classifier[0].in_features
model.classifier = nn.Sequential(
    nn.Linear(num_features, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 1),
    nn.Sigmoid()
)

### 3. 训练循环 (The Training Loop)
PyTorch 没有 model.fit()，你需要手动写循环。

In [ ]:
import torch.optim as optim

# 定义优化器和损失函数 (对应 model.compile)
criterion = nn.BCELoss() # 二分类交叉熵
optimizer = optim.RMSprop(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# 初始化用于保存 Loss 的数组
train_loss_history = []
val_loss_history = []

# 模拟 model.fit()
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        labels = labels.float().unsqueeze(1) # 适配 BCELoss 的维度

        # --- 核心五步法 ---
        optimizer.zero_grad()               # 1. 清零梯度
        outputs = model(inputs)             # 2. 前向传播
        loss = criterion(outputs, labels)   # 3. 计算 Loss
        loss.backward()                     # 4. 反向传播
        optimizer.step()                    # 5. 更新参数
        running_loss += loss.item()

    # 记录本轮平均训练损失
    epoch_train_loss = running_loss / len(train_loader)
    train_loss_history.append(epoch_train_loss)
    # --- 验证阶段 ---
    model.eval() # 开启评估模式 (关闭 Dropout 等)
    running_val_loss = 0.0

    with torch.no_grad(): # 禁用梯度计算，节省内存
        for inputs, labels in val_loader: # 假设你已经定义了 val_loader
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

    # 记录本轮平均验证损失
    epoch_val_loss = running_val_loss / len(val_loader)
    val_loss_history.append(epoch_val_loss)

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

注意‼️： 验证集确实不参与参数更新。
1. 既然验证集不影响参数，为什么要每个 Epoch 跑一遍？
- Early Stopping
- Change HyperParams
- Plot
2. 如何让验证结果 “自动化”地影响下一个 Epoch？
Scheduler（学习率调度器）： 每个epoch结束，自动调整学习率。
